# 08 — Antibiogram-conflict comparison (replicate a documented FedAvg failure condition)

Runs the **same protocol twice** on the pre-registered `antibiogram_conflict` split —
once with plain FedAvg, once with the topology-aware strategy — judged against the
criteria fixed BEFORE any run in `docs/2026-10-18-antibiogram-conflict-split.md`.
Read that doc first: it defines C1/C2/C3 and the binding falsification rules.

**Before running:**
1. Runtime → **GPU** (high-RAM helps; full 1.6M-row cohort).
2. Run the **smoke cell first** (minutes, no training). If an assert fails, STOP —
   the split lacks enough conflict on real data; fix the split BEFORE burning GPU
   hours. That is what the smoke gate is for.
3. Arms are 5 seeds x 2 arms (v2 protocol: 8 rounds x 4 epochs, hidden 128) —
   a few hours each. Each arm is one cell, so a failure loses nothing else.
4. Gates: **C1** FedAvg hurts the minority (worst_fed <= worst_local - 0.01) —
   the literature condition reproduced; **C2** Path-B gate (topo >= +0.005 on
   worst-hospital or mean); **C3** topology must RAISE the tail hospital.


In [ ]:
# 1) Get the code + deps (+ record versions — deps are unpinned, so log them every run)
!git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
!cd /content/Capstone-amr-fed && git log -1 --oneline
!pip install -q torch_geometric 'flwr[simulation]'
!pip freeze | grep -E '^(torch|torch-geometric|flwr|ray)==' || true


In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder


In [ ]:
# 3) Sanity: data resolves + imports
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'
from amr_fed.federated.run import run_fedavg, run_multiseed
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import antibiogram_conflict, partition_summary, _patient_shared_rate
print('imports OK')


In [ ]:
# 4) Load the cohort ONCE and reuse it for every arm
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame
df = load_cohort_frame()
print('rows:', f'{len(df):,}')


## Smoke gate — partition validity only (no training, minutes)

Prints per-hospital sizes + antibiograms and asserts the pre-registered validity
checks: 1:1 patient assignment, every hospital >= 3% of tests, resistance-rate
spread >= 0.15. **If an assert fails, stop here and report the output.**


In [ ]:
# 5) SMOKE: partition validity + per-hospital antibiograms (no training)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.partition import antibiogram_conflict, partition_summary, _patient_shared_rate

assign = antibiogram_conflict(df, n_clients=5)
g, niid = partition_summary(df, assign)
print(g, "\nnon-IID(wstd):", round(niid, 4))
assert assign.index.is_unique and len(assign) == df[PK].nunique(), 'not 1:1'
assert (g['n_tests'] / g['n_tests'].sum() >= 0.03).all(), 'hospital below 3% size floor'
spread = g['resist_rate'].max() - g['resist_rate'].min()
assert spread >= 0.15, f'rate spread {spread:.4f} < 0.15 — split lacks conflict'
print(f'rate spread OK: {spread:.4f}')
print('SMOKE PASS — safe to run the arms.')


## Pre-flight — SIGFPE diagnostics (run before the arms; ~2 min total)

The FedAvg/topology arms construct each Flower client inside a Ray actor via
`client_fn -> FlowerClient.__init__ -> init_model_on`. On the 2026-10-22 Colab runtime
(python 3.13.15 / torch 2.11.0+cu130 / pyg 2.8.1 / flwr 1.41.0 / ray 2.55.1) that
lazy `SAGEConv` weight init dies with **SIGFPE in `uniform_`** — but only under Flower:
the same `init_model_on` passes in the driver (local-only/pooled) and in plain Ray
actors. Inputs are provably identical (`/tmp/amr_fed_clients` is a fixed path), so the
difference is in-process state set by something in Flower's client path.

**Run the three cells below in order, then paste their output back into chat:**

- **D1** — driver env + FP-trap mask baseline (5s)
- **D2** — staged bisect: which import/preamble poisons the actor? each stage = its own
  Ray task, so a SIGFPE only kills that stage and gets reported (1 min)
- **D3** — repro-mini: the REAL Flower crash in ~40s with `[client-init]`/`[kaiming]`
  forensics on (`AMR_DEBUG_INIT=1`, set by the cell)

Once the fix lands these three can be skipped; the arms below are self-contained
(pull `adb2115`+ and run the notebook top-to-bottom).


In [ ]:
# D1) Environment + FP-trap mask baseline (driver)
import sys, ctypes
print('python', sys.version.split()[0])
import torch, torch_geometric, flwr, ray
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| pyg', torch_geometric.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__)
print('OMP_NUM_THREADS', __import__('os').environ.get('OMP_NUM_THREADS'),
      '| CUDA_VISIBLE_DEVICES', __import__('os').environ.get('CUDA_VISIBLE_DEVICES'))
_libm = ctypes.CDLL('libm.so.6')   # glibc fegetexcept = enabled FP-trap mask
def _mask():
    try: return int(_libm.fegetexcept())
    except Exception as e: return f'err {e}'
print('DRIVER fenv trap mask:', _mask(),
      '| loaded:', [m.split('.')[0] for m in sys.modules
                    if m.split('.')[0] in ('jax', 'jaxlib', 'tensorflow')])


In [ ]:
# D2) STAGED BISECT: which import/preamble poisons the actor? (each stage = its own task)
import sys, ctypes, cloudpickle, ray
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed.federated import client_app as _m
APP_BLOB = cloudpickle.dumps(_m.app)      # exactly what Flower ships to actors

ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False)

@ray.remote(num_cpus=1, num_gpus=0.18)    # same resources Flower gives clients
def stage(name):
    import os, sys, ctypes, traceback
    # Flower gives REAL actors the driver's sys.path as a PYTHONPATH runtime_env
    # (flwr raybackend.py:115-120) - plain Ray tasks do not get it. Replicate:
    sys.path.insert(0, '/content/Capstone-amr-fed/src')
    out = {'stage': name}
    libm = ctypes.CDLL('libm.so.6')
    try: out['mask0'] = int(libm.fegetexcept())
    except Exception as e: out['mask0'] = f'err {e}'
    out['preloaded'] = [m.split('.')[0] for m in sys.modules
                        if m.split('.')[0] in ('jax', 'jaxlib', 'tensorflow')]
    try:
        if name == 's0_baseline':
            import torch
            torch.empty(128, 64).uniform_(-.1, .1)
            torch.empty(0, 64).uniform_(-.1, .1)     # empty-tensor theory
        elif name == 's1_flwr_client':               # flwr stack Flower imports in actors
            import flwr.client, flwr.common, flwr.clientapp.client_app
            import flwr.client.message_handler.message_handler, flwr.app
        elif name == 's2_our_app':                   # runs module-level ClientApp(...)
            import amr_fed.federated.client_app
        elif name == 's3_unpickle_app':              # = client_fn() path in ray_actor.run
            cloudpickle.loads(APP_BLOB)
        elif name == 's4_real_init':                 # control: full real client init
            from amr_fed.federated.task import seed_client, read_run_config, \
                load_client_graph, init_model_on, DEVICE
            cfg = read_run_config(); seed_client(cfg, 0)
            out['device'] = DEVICE
            data = load_client_graph(0)
            out['graph'] = {str(k): list(v.size()) for k, v in data.x_dict.items()}
            import torch
            out['threads'] = torch.get_num_threads()
            out['env'] = {k: os.environ[k][:60] for k in os.environ
                          if any(s in k for s in ('CUDA_VISIBLE', 'OMP_', 'MKL_', 'PYTORCH'))}
            init_model_on(data, cfg)
            out['init'] = 'ok'
        elif name == 's5_import_jax':                # jaxlib appears in crash dumps
            import jax
    except BaseException:
        out['error'] = traceback.format_exc()[-700:]
    finally:                                        # capture mask even on error
        try:
            out['mask1'] = int(libm.fegetexcept())
            import torch
            torch.empty(128, 64).uniform_(-.1, .1)
            out['uniform_after'] = 'ok'
        except BaseException as e:
            out['uniform_after'] = f'FAIL {e!r}'
    return out

for nm in ['s0_baseline', 's1_flwr_client', 's2_our_app', 's3_unpickle_app',
           's4_real_init', 's5_import_jax']:
    try:
        print(ray.get(stage.remote(nm)))
    except BaseException as e:
        print({'stage': nm, 'CRASHED': str(e)[:250]})   # SIGFPE kills only this task
ray.shutdown()                                         # release Ray before any arm cell

# Read: mask != 0 at some stage -> FP traps enabled there -> fedisableexcept fix.
# s5 crashes -> jax poisons state -> same fix.
# ALL pass -> poison happens only in real app(message, context) execution -> next:
# instrument PyG kaiming_uniform inside client_app.py to log shape/fan/bound/mask.


In [ ]:
# D3) REPRO-MINI: hit the real Flower crash in ~40s with forensics on (AMR_DEBUG_INIT=1)
import os, ray, sys as _s
os.environ["AMR_DEBUG_INIT"] = "1"     # client_app prints [client-init]/[kaiming] lines
ray.shutdown()                         # fresh raylet so actors inherit the env var
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_fedavg
from amr_fed.partition import antibiogram_conflict
run_fedavg(n_clients=5, rounds=1, local_epochs=1, seed=42, df=df,
           partition_fn=lambda d, s: antibiogram_conflict(d, n_clients=5, seed=s),
           label='repro-mini', compute_pooled=False, local_only_epochs=1, hidden=128)

# READ: lines prefixed (ClientAppActor pid=...):
#   [bisect b1/b2] -> actor env dump + fresh-subprocess rc (rc!=0 => env itself poisoned)
#   [zero-init]    -> the fix engaged: weights materialised without any RNG call
#   [client-init]  -> device/threads/fenv/graph/cfg; "model OK" x5 = construction done
#   Fatal Python error / any NEW crash site -> paste it (mid-fit = training-time RNG)
# (rerun D2 first: its s4 stage needs run_config.json, now on disk)
# Paste ALL (ClientAppActor ...) lines back into chat.


## Arms (5 seeds each, v2 protocol: 8 rounds x 4 epochs = 32 budget, hidden 128)


In [ ]:
# 6) ANTIBIOGRAM-CONFLICT, FedAvg arm (5 seeds)
import os, ray
os.environ["AMR_DEBUG_INIT"] = "1"   # actors print [client-init]/[kaiming] SIGFPE forensics
ray.shutdown()                        # fresh raylet so workers inherit the env var
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import antibiogram_conflict
ab_fed = run_multiseed(seeds=(42, 43, 44, 45, 46), df=df, label='antibiogram/FedAvg',
                       partition_fn=lambda d, s: antibiogram_conflict(d, n_clients=5, seed=s),
                       n_clients=5, rounds=8, local_epochs=4,
                       local_only_epochs=32, hidden=128)


In [ ]:
# 7) ANTIBIOGRAM-CONFLICT, topology arm (same protocol, distinctiveness T=1.0)
import os, ray
os.environ["AMR_DEBUG_INIT"] = "1"   # actors print [client-init]/[kaiming] SIGFPE forensics
ray.shutdown()                        # fresh raylet so workers inherit the env var
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.partition import antibiogram_conflict
ab_topo = run_multiseed(seeds=(42, 43, 44, 45, 46), df=df, label='antibiogram/TOPO',
                        partition_fn=lambda d, s: antibiogram_conflict(d, n_clients=5, seed=s),
                        n_clients=5, rounds=8, local_epochs=4,
                        local_only_epochs=32, hidden=128, strategy='topology',
                        topo_mode='distinctiveness', topo_temperature=1.0)


## Verdict — pre-registered C1/C2/C3

Run after BOTH arms. Prints the three criteria and the per-seed tail-hospital F1s,
saves JSON to Drive, and prints a COPY block — paste it back into chat.


In [ ]:
# 8) VERDICT: pre-registered C1/C2/C3 (docs/2026-10-18-antibiogram-conflict-split.md)
import json
import numpy as np

def _tail_f1(res, key):
    vals = []
    for r in res['runs']:
        pc = r.get(key)
        if not pc:
            continue
        vals.append(pc[max(pc, key=lambda x: int(x))] if isinstance(pc, dict) else pc[-1])
    return vals

d_mean = round(ab_topo['fedavg_best'][0] - ab_fed['fedavg_best'][0], 4)
d_worst = round(ab_topo['worst_fed'][0] - ab_fed['worst_fed'][0], 4)
c1 = ab_fed['worst_fed'][0] <= ab_fed['worst_local'][0] - 0.01
c2 = (d_worst >= 0.005) or (d_mean >= 0.005)

print(f"fedavg  best {ab_fed['fedavg_best']} | worst {ab_fed['worst_fed']} "
      f"| worst_local {ab_fed['worst_local']}")
print(f"topo    best {ab_topo['fedavg_best']} | worst {ab_topo['worst_fed']}")
print(f"C1 FedAvg hurts minority (worst_fed <= worst_local - 0.01): "
      f"{'PASS' if c1 else 'FAIL'}")
print(f"C2 Path-B gate (delta_mean {d_mean} | delta_worst {d_worst}): "
      f"{'PASS' if c2 else 'FAIL'}")

tail_f = _tail_f1(ab_fed, 'fedavg_per_client')
tail_t = _tail_f1(ab_topo, 'fedavg_per_client')
tail_l = _tail_f1(ab_fed, 'local_only_per_client')
tf = tt = tl = None
if tail_f and tail_t:
    tf, tt = round(float(np.mean(tail_f)), 4), round(float(np.mean(tail_t)), 4)
    tl = round(float(np.mean(tail_l)), 4) if tail_l else None
    print(f"C3 tail hospital F1: local {tl} | fedavg {tf} | topo {tt} -> "
          f"{'PASS' if tt > tf else 'FAIL'} (topo must RAISE the tail)")

payload = {'C1': bool(c1), 'C2': bool(c2), 'delta_mean': d_mean, 'delta_worst': d_worst,
           'tail_local': tl, 'tail_fedavg': tf, 'tail_topo': tt,
           'tail_fedavg_per_seed': tail_f, 'tail_topo_per_seed': tail_t,
           'fed': {'best': ab_fed['fedavg_best'], 'worst': ab_fed['worst_fed'],
                   'worst_local': ab_fed['worst_local'], 'final': ab_fed['fedavg_final']},
           'topo': {'best': ab_topo['fedavg_best'], 'worst': ab_topo['worst_fed'],
                    'final': ab_topo['fedavg_final']}}
path = '/content/drive/MyDrive/amr_antibiogram_verdict.json'
with open(path, 'w') as f:
    json.dump(payload, f, default=str, indent=1)
print('saved', path)
print('=== COPY EVERYTHING BELOW THIS LINE BACK TO CHAT ===')
print(json.dumps(payload, default=str))
print('=== END ===')
